# Identificação de Raça de Gatos com ResNet50

Compara o "DNA" (vetor de features) de uma foto de teste com fotos de referência de 5 raças:
**Maine Coon, Persa, Ragdoll, Siamês, Sphynx**

In [ ]:
import numpy as np
import os
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image

base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')

def extrair_dna_da_foto(caminho_da_imagem):
    img = image.load_img(caminho_da_imagem, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    features = base_model.predict(x, verbose=0)
    return features

print("IA pronta para analisar!")

In [ ]:
import glob

RACAS = ["mainecoon", "persa", "ragdoll", "siames", "sphynx"]
EXTENSOES = ("*.png", "*.jpg", "*.jpeg", "*.webp")
BASE_DIR = "fotos_gatos"

banco_referencia = {}

for raca in RACAS:
    pasta = os.path.join(BASE_DIR, raca)
    arquivos = []
    for ext in EXTENSOES:
        arquivos.extend(glob.glob(os.path.join(pasta, ext)))

    if not arquivos:
        print(f"[AVISO] Nenhuma imagem encontrada em {pasta}/")
        continue

    foto = arquivos[0]
    banco_referencia[raca] = {
        "dna": extrair_dna_da_foto(foto),
        "foto": foto
    }
    print(f"{raca}: {os.path.basename(foto)} carregado")

print(f"\nMemoria carregada com {len(banco_referencia)} racas!")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def descobrir_e_mostrar_resultado(caminho_teste):
    dna_novo = extrair_dna_da_foto(caminho_teste)

    distancias = {}
    for nome, dados in banco_referencia.items():
        dist = np.linalg.norm(dados["dna"] - dna_novo)
        distancias[nome] = dist
        print(f"Distancia para {nome}: {dist:.2f}")

    vencedor = min(distancias, key=distancias.get)

    fig, ax = plt.subplots(1, 2, figsize=(10, 5))

    img_teste = Image.open(caminho_teste)
    ax[0].imshow(img_teste)
    ax[0].set_title(f"FOTO TESTE\n({os.path.basename(caminho_teste)})", fontsize=12, fontweight='bold')
    ax[0].axis('off')

    img_ref = Image.open(banco_referencia[vencedor]["foto"])
    ax[1].imshow(img_ref)
    ax[1].set_title(
        f"IA IDENTIFICOU COMO: {vencedor.upper()}\n(Foto de Referencia)",
        fontsize=12,
        fontweight='bold',
        color='green'
    )
    ax[1].axis('off')

    plt.tight_layout()
    plt.show()

pasta_test = os.path.join(BASE_DIR, "test")
arquivos_test = []
for ext in EXTENSOES:
    arquivos_test.extend(glob.glob(os.path.join(pasta_test, ext)))

if not arquivos_test:
    print("Nenhuma imagem encontrada em fotos_gatos/test/")
else:
    for foto_teste in arquivos_test:
        print(f"\n--- Testando: {os.path.basename(foto_teste)} ---")
        descobrir_e_mostrar_resultado(foto_teste)